<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os
import glob
import pandas as pd

print("Current folder:", os.getcwd())

# Search for the dataset
matches = glob.glob("/content/**/content_refresh_anonymized.csv", recursive=True)

print("Dataset files found:", len(matches))

if matches:
    print("Dataset found at:")
    for path in matches:
        print(path)
else:
    print("NO DATASET FOUND")

Current folder: /content
Dataset files found: 1
Dataset found at:
/content/AIML/data/raw/content_refresh_anonymized.csv


In [16]:
!git clone https://github.com/vaishnavikabbe/AIML.git /content/AIML

fatal: destination path '/content/AIML' already exists and is not an empty directory.


In [17]:
import glob
import pandas as pd

matches = glob.glob(
    "/content/AIML/**/content_refresh_anonymized.csv",
    recursive=True
)

print("Dataset files found:", len(matches))

for path in matches:
    print(path)

Dataset files found: 1
/content/AIML/data/raw/content_refresh_anonymized.csv


In [18]:
DATA_PATH = matches[0]

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Path:", DATA_PATH)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully!
Path: /content/AIML/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [19]:
print("ALL 44 COLUMNS")
print("=" * 60)

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

ALL 44 COLUMNS
1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The action queue prioritizes pages that show stronger observed signals for content refresh review. Each page receives a reason code so that a human reviewer can understand why it was selected.

Reason codes:
- HIGH_IMPRESSIONS: The page has relatively high observed search impressions, so a refresh could affect an important page.
- DECLINING_TREND: The page is observed as declining and may deserve review.
- LOW_CONTENT_DEPTH: The page has relatively low word count compared with other pages.
- MULTIPLE_SIGNALS: More than one review signal is present.

The ranking is intended as decision-support, not as an automatic instruction to update a page.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 1
# Build a transparent action queue from the actual dataset

import pandas as pd
import numpy as np

work = df.copy()

# Start with zero score
work["action_score"] = 0.0

# Store reasons separately
work["reason_codes"] = ""

# ---------- Signal 1: declining trend ----------
if "trend_direction" in work.columns:
    declining = (
        work["trend_direction"]
        .astype(str)
        .str.lower()
        .str.strip()
        .eq("declining")
    )

    work.loc[declining, "action_score"] += 2
    work.loc[declining, "reason_codes"] = (
        work.loc[declining, "reason_codes"] + "DECLINING_TREND;"
    )

# ---------- Signal 2: high observed impressions ----------
if "impressions_90d" in work.columns:
    impression_median = work["impressions_90d"].median()

    high_impressions = (
        work["impressions_90d"] >= impression_median
    )

    work.loc[high_impressions, "action_score"] += 1
    work.loc[high_impressions, "reason_codes"] = (
        work.loc[high_impressions, "reason_codes"]
        + "HIGH_IMPRESSIONS;"
    )

# ---------- Signal 3: relatively low word count ----------
if "word_count" in work.columns:
    word_median = work["word_count"].median()

    low_word_count = (
        work["word_count"] < word_median
    )

    work.loc[low_word_count, "action_score"] += 1
    work.loc[low_word_count, "reason_codes"] = (
        work.loc[low_word_count, "reason_codes"]
        + "LOW_CONTENT_DEPTH;"
    )

# Give pages with no specific signal a general code
work.loc[
    work["reason_codes"] == "",
    "reason_codes"
] = "GENERAL_REVIEW"

# Remove trailing semicolons
work["reason_codes"] = work["reason_codes"].str.rstrip(";")

# Rank pages
work = work.sort_values(
    ["action_score"],
    ascending=False
).reset_index(drop=True)

work["rank"] = range(1, len(work) + 1)

print("Action queue created successfully.")
print("Pages ranked:", len(work))

print("\nTop 20:")
display(
    work[
        [
            "rank",
            "action_score",
            "reason_codes"
        ]
        + [
            c for c in [
                "content_id",
                "trend_direction",
                "impressions_90d",
                "word_count"
            ]
            if c in work.columns
        ]
    ].head(20)
)

DATA_PATH = matches[0]
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# Work on a copy
work = df.copy()

# Create a simple action score using observed fields
work["action_score"] = 0.0
work["reason_code"] = "REVIEW"

# Declining trend
if "trend_direction" in work.columns:
    declining = work["trend_direction"].astype(str).str.lower().eq("declining")
    work.loc[declining, "action_score"] += 2
    work.loc[declining, "reason_code"] = "DECLINING_TREND"

# High impressions
if "impressions_90d" in work.columns:
    impression_threshold = work["impressions_90d"].median()
    high_impressions = work["impressions_90d"] >= impression_threshold
    work.loc[high_impressions, "action_score"] += 1

# Low content depth
if "word_count" in work.columns:
    word_threshold = work["word_count"].median()
    low_content = work["word_count"] < word_threshold
    work.loc[low_content, "action_score"] += 1

# Multiple signals
if "trend_direction" in work.columns and "word_count" in work.columns:
    multiple = declining & low_content
    work.loc[multiple, "reason_code"] = "MULTIPLE_SIGNALS"

# Rank
work = work.sort_values(
    ["action_score"],
    ascending=False
).reset_index(drop=True)

work["rank"] = range(1, len(work) + 1)

print("\nTop 10 recommended pages for review:")
display(
    work[
        ["rank", "action_score", "reason_code"]
        + [c for c in ["trend_direction", "impressions_90d", "word_count"]
           if c in work.columns]
    ].head(10)
)

Action queue created successfully.
Pages ranked: 30000

Top 20:


,rank,action_score,reason_codes,content_id,trend_direction,impressions_90d,word_count
0,1,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_1956a8699d64,up,1606,2751.0
1,2,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_fc0f5616de45,down,3437,2411.0
2,3,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_eb7046437af1,stable,1235,2796.0
3,4,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_d8e1b5fb9606,stable,2781,1870.0
4,5,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_c73b8bef0152,down,1050,2463.0
5,6,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_8b5aab518cbb,down,1313,1561.0
6,7,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_7a09da8ecc46,down,1761,2725.0
7,8,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_f670e8a58608,down,778,2873.0
8,9,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_e4c789b3ac31,down,4066,1497.0
9,10,2.0,HIGH_IMPRESSIONS;LOW_CONTENT_DEPTH,content_2d12a2ae6fc7,down,10428,2359.0


Dataset loaded successfully.
Rows: 30000
Columns: 44

Top 10 recommended pages for review:


,rank,action_score,reason_code,trend_direction,impressions_90d,word_count
0,1,2.0,REVIEW,up,1606,2751.0
1,2,2.0,REVIEW,down,3437,2411.0
2,3,2.0,REVIEW,stable,1235,2796.0
3,4,2.0,REVIEW,stable,2781,1870.0
4,5,2.0,REVIEW,down,1050,2463.0
5,6,2.0,REVIEW,down,1313,1561.0
6,7,2.0,REVIEW,down,1761,2725.0
7,8,2.0,REVIEW,down,778,2873.0
8,9,2.0,REVIEW,down,4066,1497.0
9,10,2.0,REVIEW,down,10428,2359.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The action queue is intended for SEO or content team members who need to decide which pages should receive human review first. It summarizes observed search-performance and content signals into a ranked review list.

The queue does not decide whether a page should definitely be changed. It does not prove that refreshing a page will improve search performance, and it should not be treated as a prediction of search-engine rankings.

The recommendations are directional decision-support based on the available anonymized dataset.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic checks for the recommendation queue

# Verify that the queue is usable

# Section 2: Check the intended-use output

# Confirm that the ranking created in Section 1 exists
required_columns = ["rank", "action_score", "reason_code"]

missing_columns = [
    col for col in required_columns
    if col not in work.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns from the ranking: {missing_columns}. "
        "Please run Section 1 first."
    )

print("Intended-use checks")
print("-------------------")

print("Pages in review queue:", len(work))
print("Minimum rank:", work["rank"].min())
print("Maximum rank:", work["rank"].max())

print("\nReason codes:")
print(work["reason_code"].value_counts())

print("\nTop 10 rows in the review queue:")

display(
    work[
        [
            "rank",
            "action_score",
            "reason_code"
        ]
    ]
    .head(10)
)

Intended-use checks
-------------------
Pages in review queue: 30000
Minimum rank: 1
Maximum rank: 30000

Reason codes:
reason_code
REVIEW    30000
Name: count, dtype: int64

Top 10 rows in the review queue:


,rank,action_score,reason_code
0,1,2.0,REVIEW
1,2,2.0,REVIEW
2,3,2.0,REVIEW
3,4,2.0,REVIEW
4,5,2.0,REVIEW
5,6,2.0,REVIEW
6,7,2.0,REVIEW
7,8,2.0,REVIEW
8,9,2.0,REVIEW
9,10,2.0,REVIEW


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

The ranked queue is a starting point for human review, not an automatic instruction to change a page. Before acting, a reviewer should check the page's actual content, search intent, relevance, freshness, and whether the observed signals make sense in context.

The system should never automatically publish, delete, rewrite, or make major content decisions based only on the score. It should also not treat the ranking as proof that a refresh will improve search performance.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Human review checks

# Make sure Section 1 has created the ranked queue
required_columns = ["rank", "action_score", "reason_code"]

missing_columns = [
    col for col in required_columns
    if col not in work.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}. "
        "Please run Section 1 first."
    )

# Select the top 20 pages for human review
top20 = work.head(20).copy()

print("Top 20 pages requiring human review:")
print("--------------------------------------")

display(
    top20[
        [
            "rank",
            "action_score",
            "reason_code"
        ]
        + [
            col for col in [
                "trend_direction",
                "impressions_90d",
                "word_count"
            ]
            if col in top20.columns
        ]
    ]
)

print("\nHuman review is required before any action is taken.")

print("\nNo-go list:")
print("1. Do not automatically publish content changes.")
print("2. Do not automatically delete or rewrite pages.")
print("3. Do not assume a high score means a refresh will improve performance.")
print("4. Do not use the score as proof of causation or search-engine ranking.")

Top 20 pages requiring human review:
--------------------------------------


,rank,action_score,reason_code,trend_direction,impressions_90d,word_count
0,1,2.0,REVIEW,up,1606,2751.0
1,2,2.0,REVIEW,down,3437,2411.0
2,3,2.0,REVIEW,stable,1235,2796.0
3,4,2.0,REVIEW,stable,2781,1870.0
4,5,2.0,REVIEW,down,1050,2463.0
5,6,2.0,REVIEW,down,1313,1561.0
6,7,2.0,REVIEW,down,1761,2725.0
7,8,2.0,REVIEW,down,778,2873.0
8,9,2.0,REVIEW,down,4066,1497.0
9,10,2.0,REVIEW,down,10428,2359.0



Human review is required before any action is taken.

No-go list:
1. Do not automatically publish content changes.
2. Do not automatically delete or rewrite pages.
3. Do not assume a high score means a refresh will improve performance.
4. Do not use the score as proof of causation or search-engine ranking.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The recommendations may become stale if the distribution of search-performance or content signals changes over time. They may also become less useful if the relationship between these signals and the observed trend direction changes.

I would review the scoring approach when new data becomes available, when important fields change meaning or availability, or when the ranked recommendations no longer agree well with human review. A substantial change in the data distribution or a sustained drop in useful ranking performance would be a reason to investigate and potentially update the approach.

Retraining or rebuilding the model should only happen after checking the new data and confirming that the previous approach is no longer suitable.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Basic monitoring checks

print("Monitoring checks")
print("-----------------")

# Check the size of the current dataset
print("Number of pages:", len(work))

# Check score distribution
print("\nAction score summary:")
print(work["action_score"].describe().round(2))

# Check reason-code distribution
print("\nReason-code distribution:")
print(work["reason_code"].value_counts())

# Check trend distribution if available
if "trend_direction" in work.columns:
    print("\nTrend distribution:")
    print(work["trend_direction"].value_counts(dropna=False))

# Check missing values in important fields
monitor_columns = [
    col for col in [
        "action_score",
        "reason_code",
        "trend_direction",
        "impressions_90d",
        "word_count"
    ]
    if col in work.columns
]

print("\nMissing values in monitored fields:")
print(work[monitor_columns].isna().sum())

print("\nMonitoring complete.")
print("A change in these distributions or repeated disagreement with human review")
print("would be a reason to investigate whether the scoring approach is still useful.")

Monitoring checks
-----------------
Number of pages: 30000

Action score summary:
count    30000.00
mean         0.87
std          0.65
min          0.00
25%          0.00
50%          1.00
75%          1.00
max          2.00
Name: action_score, dtype: float64

Reason-code distribution:
reason_code
REVIEW    30000
Name: count, dtype: int64

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing values in monitored fields:
action_score          0
reason_code           0
trend_direction       0
impressions_90d       0
word_count         7699
dtype: int64

Monitoring complete.
A change in these distributions or repeated disagreement with human review
would be a reason to investigate whether the scoring approach is still useful.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked action queue is exported as a CSV so it can be reused in the final analysis and paper. The export contains the ranking, action score, reason code, and relevant observed signals used for human review.

The exported file is a decision-support artifact. It does not represent guaranteed recommendations or causal conclusions.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5: Export the ranked queue for the paper

import os
import pandas as pd

# Create the output directory
OUTPUT_DIR = "/content/AIML/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check that the ranked dataframe exists
if "work" not in globals():
    raise ValueError(
        "The ranked dataframe 'work' does not exist. "
        "Please run Section 1 first."
    )

# Columns to export
export_columns = [
    col for col in [
        "rank",
        "action_score",
        "reason_code",
        "content_id",
        "trend_direction",
        "impressions_90d",
        "word_count",
        "search_volume"
    ]
    if col in work.columns
]

# Create final export
paper_queue = work[export_columns].copy()

# Save CSV
output_file = os.path.join(
    OUTPUT_DIR,
    "content_action_queue.csv"
)

paper_queue.to_csv(
    output_file,
    index=False
)

print("Export completed successfully!")
print("File:", output_file)
print("Rows exported:", len(paper_queue))
print("Columns exported:", len(paper_queue.columns))

print("\nExported columns:")
print(list(paper_queue.columns))

print("\nTop 10 exported rows:")
display(paper_queue.head(10))

Export completed successfully!
File: /content/AIML/work/outputs/content_action_queue.csv
Rows exported: 30000
Columns exported: 8

Exported columns:
['rank', 'action_score', 'reason_code', 'content_id', 'trend_direction', 'impressions_90d', 'word_count', 'search_volume']

Top 10 exported rows:


,rank,action_score,reason_code,content_id,trend_direction,impressions_90d,word_count,search_volume
0,1,2.0,REVIEW,content_1956a8699d64,up,1606,2751.0,10.0
1,2,2.0,REVIEW,content_fc0f5616de45,down,3437,2411.0,30.0
2,3,2.0,REVIEW,content_eb7046437af1,stable,1235,2796.0,20.0
3,4,2.0,REVIEW,content_d8e1b5fb9606,stable,2781,1870.0,10.0
4,5,2.0,REVIEW,content_c73b8bef0152,down,1050,2463.0,10.0
5,6,2.0,REVIEW,content_8b5aab518cbb,down,1313,1561.0,10.0
6,7,2.0,REVIEW,content_7a09da8ecc46,down,1761,2725.0,0.0
7,8,2.0,REVIEW,content_f670e8a58608,down,778,2873.0,110.0
8,9,2.0,REVIEW,content_e4c789b3ac31,down,4066,1497.0,10.0
9,10,2.0,REVIEW,content_2d12a2ae6fc7,down,10428,2359.0,10.0


In [25]:
# Verify that the exported file exists

if os.path.exists(output_file):
    print("SUCCESS: Export file exists.")
    print(output_file)
else:
    print("ERROR: Export file was not created.")

SUCCESS: Export file exists.
/content/AIML/work/outputs/content_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.